# Building a Research Assistant with Web + X Search

What do reputable news sources say about a topic? What does the crowd on X think? And where do those two stories diverge?

In this guide, we'll build a research assistant that cross-references reporting from vetted news outlets with public discourse on X, then produces a structured "divergence briefing" that clusters claims into five categories: consensus, X-ahead-of-press, press-ahead-of-X, X-only, and press-only.

This takes advantage of Grok's built-in web search and X search as native tools, with domain filtering to scope web results to trusted outlets and inline citations linking every claim back to its source.

### What we'll use
- Web search with domain filtering (`allowed_domains`) to restrict results to trusted outlets
- X search to capture real-time public discourse
- Inline citations linking every claim back to its source
- Structured output with Pydantic models to parse the final analysis
- The native xai-sdk

### Table of Contents
- [Setup](#setup)
- [Act 1: Web Search with Domain Filtering](#act-1-web-search-with-domain-filtering)
- [Interlude: Filtered vs. Unfiltered](#interlude-filtered-vs-unfiltered)
- [Act 2: X Search for Public Discourse](#act-2-x-search-for-public-discourse)
- [Act 3: The Divergence Analysis](#act-3-the-divergence-analysis)
- [Structured Output](#structured-output)
- [Putting It All Together](#putting-it-all-together)
- [Conclusion](#conclusion)

## Setup

We use the native `xai-sdk` rather than the OpenAI compatibility layer, since `web_search` domain filtering, `x_search`, and inline citations are first-class features of the native SDK.

All you need is an xAI API key.

In [1]:
%pip install -q xai-sdk python-dotenv

In [2]:
import os

from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import system, user
from xai_sdk.tools import web_search, x_search

load_dotenv()

XAI_API_KEY = os.environ.get("XAI_API_KEY")
if not XAI_API_KEY:
    raise ValueError("XAI_API_KEY is not set. Add it to your .env file or export it in your shell.")

client = Client(api_key=XAI_API_KEY)

MODEL = "grok-4.20-reasoning"

## Act 1: Web Search with Domain Filtering

Grok's `web_search` tool lets you restrict results to specific domains using `allowed_domains` (max 5 per call). Instead of hoping the model finds good sources, you tell it where to look.

We'll define curated domain lists for different research contexts, then run a search scoped to major news outlets.

In [3]:
# Curated domain lists for different research contexts
NEWS_DOMAINS = ["reuters.com", "apnews.com", "bbc.com", "ft.com", "wsj.com"]
TECH_DOMAINS = ["techcrunch.com", "arstechnica.com", "theverge.com", "wired.com"]

In [4]:
TOPIC = "Space-based data centres and the future of orbital computing infrastructure"

chat = client.chat.create(
    model=MODEL,
    tools=[web_search(allowed_domains=NEWS_DOMAINS)],
    include=["inline_citations"],
)
chat.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims from these sources, with citations. Be concise."
))

response = None
for response, chunk in chat.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**

Key

 claims

 from

 recent

...

 in

 the

 near

 term

.

Every claim is backed by an inline citation linking to the original article. Let's inspect those citations programmatically:

In [5]:
print("Web search citations:")
for citation in response.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

Web search citations:
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/business/aerospace-defense/musks-mega-merger-spacex-xai-bets-sci-fi-future-data-centers-space-2026-02-04/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/business/aerospace-defense/musks-mega-merger-spacex-xai-bets-sci-fi-future-data-centers-space-2026-02-04/
  https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  https://www.reuters.com/business/aerospace-defense/spacexs-orbital-data-centers-could-face-same-hurdles-microsofts-abandoned-2026-04-01/
  https://www.reuters.com/business/aerospace-defense/amazons-

Notice every source is from our `NEWS_DOMAINS` list.

## Interlude: Filtered vs. Unfiltered

What happens if we run the same query without domain filtering? The model is free to pull from any source. This isn't a controlled experiment (ranking, freshness, and source availability all vary between calls), but it illustrates why scoping your sources matters for research.

In [6]:
chat_unfiltered = client.chat.create(
    model=MODEL,
    tools=[web_search()],
    include=["inline_citations"],
)
chat_unfiltered.append(user(
    f"Search for the latest reporting on: {TOPIC}. "
    "Summarize the 3-5 most important claims with citations. Be concise."
))

response_unfiltered = None
for response_unfiltered, chunk in chat_unfiltered.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**

Key

 claims

 from

 recent

...

202

5

 research

 update

.**

In [7]:
print("\nUnfiltered citations:")
for citation in response_unfiltered.inline_citations:
    if citation.HasField("web_citation"):
        print(f"  {citation.web_citation.url}")

print(f"\nFiltered: {len(response.inline_citations)} citations (all from vetted outlets)")
print(f"Unfiltered: {len(response_unfiltered.inline_citations)} citations (mixed sources)")


Unfiltered citations:
  https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://research.google/blog/exploring-a-space-based-scalable-ai-infrastructure-system-design/
  https://www.technologyreview.com/2026/04/03/1135073/four-things-wed-need-to-put-data-centers-in-space/
  https://www.npr.org/2026/04/03/nx-s1-5718416/ai-data-centers-in-space-spacex-elon-musk
  https://www.technologyreview.com/2026/04/03/1135073/four-things-wed-need-to-put-data-centers-in-space/

Filtered: 9 citations (all from vetted outlets)
Unfiltered: 6 citations (mixed sources)


Both produce useful summaries, but the filtered version gives you confidence in where the information came from.

## Act 2: X Search for Public Discourse

Now let's see what people are actually saying. Grok's `x_search` tool searches X directly, with optional handle filtering and date ranges.

In [8]:
chat_x = client.chat.create(
    model=MODEL,
    tools=[x_search()],
    include=["inline_citations"],
)
chat_x.append(user(
    f"Search X for what people are saying about: {TOPIC}. "
    "Summarize the 3-5 key themes in public sentiment, with citations to specific posts. Be concise."
))

response_x = None
for response_x, chunk in chat_x.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

**

Key

 themes

 in

 public

...

5

–

202

6

.

In [9]:
print("X search citations:")
for citation in response_x.inline_citations:
    if citation.HasField("x_citation"):
        print(f"  {citation.x_citation.url}")

X search citations:
  https://x.com/SpaceComputerIO/status/2040746310417187112
  https://x.com/kimmonismus/status/1980945551995863217
  https://x.com/renaldbarnett/status/2040189083071471955
  https://x.com/niccruzpatane/status/2020175251837985052
  https://x.com/kimmonismus/status/1980945551995863217
  https://x.com/venturemanny/status/2038684453946405096
  https://x.com/SpaceComputerIO/status/2040746310417187112
  https://x.com/Andercot/status/1981465914400002550
  https://x.com/rmcentush/status/1985787187556991364
  https://x.com/aaronburnett/status/2029686439546605591
  https://x.com/Andercot/status/1981465914400002550
  https://x.com/Gykiwi03/status/2040536891192479886


Compare the tone and content with Act 1. You may notice that press reporting tends to balance opportunity against economic and technical skepticism, while X discussion often leans more optimistic or speculative. The overlap and divergence between those perspectives is what we'll formalize next.

## Act 3: The Divergence Analysis

We take the full outputs from both searches and ask Grok to reconcile them, clustering every claim into one of five categories:

1. CONSENSUS: claims both sources agree on
2. X_AHEAD_OF_PRESS: claims appearing on X but not yet in press
3. PRESS_AHEAD_OF_X: claims in press but not discussed on X
4. X_ONLY: claims unique to X with no press corroboration
5. PRESS_ONLY: claims unique to press with no social discussion

> **Caveat:** These categories reflect what each source chose to highlight, not which covered it first. The model can't verify temporal order, so read the output as a map of framing differences, not a timeline.

The three-pass architecture:
- Pass 1 (already done): Web search with domain filtering
- Pass 2 (already done): X search
- Pass 3 (below): A reconciliation call with no tools, just analysis

In [10]:
from typing import Literal

from pydantic import BaseModel

Category = Literal[
    "CONSENSUS",
    "X_AHEAD_OF_PRESS",
    "PRESS_AHEAD_OF_X",
    "X_ONLY",
    "PRESS_ONLY",
]


class Claim(BaseModel):
    text: str
    source: str
    category: Category
    url: str


class ClaimCluster(BaseModel):
    category: Category
    claims: list[Claim]


class DivergenceBrief(BaseModel):
    topic: str
    clusters: list[ClaimCluster]
    executive_summary: str

In [11]:
RECONCILIATION_PROMPT = """You are a research analyst. Given web search findings from \
reputable sources and X/social media findings on the same topic, cluster the claims \
into exactly five categories:
1. CONSENSUS — claims both sources agree on
2. X_AHEAD_OF_PRESS — claims appearing on X but not yet in press
3. PRESS_AHEAD_OF_X — claims in press but not discussed on X
4. X_ONLY — claims unique to X with no press corroboration
5. PRESS_ONLY — claims unique to press with no social discussion

For each claim, note the source ("Press" or "X"), category, and the url of the \
original source that supports it. Be thorough: extract every distinct claim from \
both inputs. Only include specific factual or analytical claims. Exclude \
meta-observations about sentiment or tone (e.g. "discussions were optimistic"). \
Output as JSON matching the provided schema."""


def _format_findings(response, label: str) -> str:
    """Format response content with its citation URLs for reconciliation."""
    lines = [response.content, f"\n### {label} Sources"]
    for citation in response.inline_citations:
        if citation.HasField("web_citation"):
            lines.append(citation.web_citation.url)
        elif citation.HasField("x_citation"):
            lines.append(citation.x_citation.url)
    return "\n".join(lines)


# Pass 1 and 2 outputs become the context for Pass 3
web_context = _format_findings(response, "Press")
x_context = _format_findings(response_x, "X")

chat_reconcile = client.chat.create(
    model=MODEL,
    response_format=DivergenceBrief,
)
chat_reconcile.append(system(RECONCILIATION_PROMPT))
chat_reconcile.append(user(
    f"## Web Search Findings\n{web_context}\n\n## X Search Findings\n{x_context}"
))

response_reconcile = None
for response_reconcile, chunk in chat_reconcile.stream():
    if chunk.content:
        print(chunk.content, end="", flush=True)

{


 "

topic

":

 "

...

 and

 regulatory

 hurdles

."


}

## Structured Output

The reconciliation call used `response_format=DivergenceBrief` to get structured JSON. Let's parse it into our Pydantic model and display it cleanly.

In [12]:
brief = DivergenceBrief.model_validate_json(response_reconcile.content)

print(f"Topic: {brief.topic}")
print(f"{'=' * 60}")
for cluster in brief.clusters:
    print(f"\n{cluster.category} ({len(cluster.claims)} claims)")
    print("-" * 40)
    for claim in cluster.claims:
        print(f"  [{claim.source}] {claim.text}")
        print(f"    {claim.url}")

print(f"\n{'=' * 60}")
print(f"Executive Summary:\n{brief.executive_summary}")

Topic: Orbital Data Centers and Space-Based AI Computing

CONSENSUS (16 claims)
----------------------------------------
  [Press] Orbital data centers provide constant 24/7 solar power with no atmosphere or weather interference
    https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  [X] Orbital data centers provide constant 24/7 solar power with no atmosphere or weather interference
    https://x.com/SpaceComputerIO/status/2040746310417187112
  [Press] Orbital data centers enable passive heat dissipation directly into space vacuum via radiators or radiative cooling
    https://www.reuters.com/sustainability/climate-energy/why-does-elon-musk-want-put-ai-data-centers-space-2026-01-29/
  [X] Orbital data centers enable passive heat dissipation directly into space vacuum via radiators or radiative cooling
    https://x.com/renaldbarnett/status/2040189083071471955
  [Press] Orbital data centers reduce strain on Earth’s power

## Putting It All Together

Let's wrap the three-pass flow into a reusable function so we can run it on any topic.

**Cost note:** Each call to `research()` makes 3 API calls (web search, X search, reconciliation). A single call is inexpensive, but costs vary with response length and model pricing changes. See [pricing](https://docs.x.ai/docs/models) for current rates.

In [13]:
def research(topic: str, web_domains: list[str] | None = None) -> DivergenceBrief:
    """Run a three-pass divergence analysis on any topic.

    Args:
        topic: The research subject.
        web_domains: Optional list of allowed domains for web search.
            Defaults to NEWS_DOMAINS if None. Max 5 per xAI docs.
    """
    domains = NEWS_DOMAINS if web_domains is None else web_domains
    if len(domains) > 5:
        raise ValueError(f"allowed_domains supports at most 5 entries, got {len(domains)}")

    # Pass 1: Web search
    chat_web = client.chat.create(
        model=MODEL,
        tools=[web_search(allowed_domains=domains)],
        include=["inline_citations"],
    )
    chat_web.append(user(
        f"Search for the latest reporting on: {topic}. "
        "Summarize the 3-5 most important claims with citations. Be concise."
    ))
    resp_web = None
    for resp_web, chunk in chat_web.stream():
        pass
    if not resp_web or not resp_web.content:
        raise RuntimeError(f"Web search returned no results for: {topic}")

    # Pass 2: X search
    chat_social = client.chat.create(
        model=MODEL,
        tools=[x_search()],
        include=["inline_citations"],
    )
    chat_social.append(user(
        f"Search X for what people are saying about: {topic}. "
        "Summarize the 3-5 key themes in public sentiment, with citations. Be concise."
    ))
    resp_social = None
    for resp_social, chunk in chat_social.stream():
        pass
    if not resp_social or not resp_social.content:
        raise RuntimeError(f"X search returned no results for: {topic}")

    # Pass 3: Reconciliation with structured output
    web_context = _format_findings(resp_web, "Press")
    x_context = _format_findings(resp_social, "X")

    chat_analysis = client.chat.create(
        model=MODEL,
        response_format=DivergenceBrief,
    )
    chat_analysis.append(system(RECONCILIATION_PROMPT))
    chat_analysis.append(user(
        f"## Web Search Findings\n{web_context}"
        f"\n\n## X Search Findings\n{x_context}"
    ))
    resp_analysis = None
    for resp_analysis, chunk in chat_analysis.stream():
        pass
    if not resp_analysis or not resp_analysis.content:
        raise RuntimeError("Reconciliation returned no results")

    return DivergenceBrief.model_validate_json(resp_analysis.content)

Let's test it on a couple of different topics. Results depend on what's being discussed when you run the notebook. If outputs look thin, try a more trending topic.

In [14]:
topics = [
    "Global semiconductor supply chain shifts and chip manufacturing reshoring",
]

for topic in topics:
    print(f"\n{'=' * 60}")
    print(f"Researching: {topic}")
    print(f"{'=' * 60}")

    brief = research(topic)

    print(f"\nTopic: {brief.topic}")
    for cluster in brief.clusters:
        print(f"  {cluster.category}: {len(cluster.claims)} claims")
    print(f"\nSummary: {brief.executive_summary}\n")


Researching: Global semiconductor supply chain shifts and chip manufacturing reshoring



Topic: US Semiconductor Reshoring and Supply Chain Diversification
  CONSENSUS: 7 claims
  X_AHEAD_OF_PRESS: 3 claims
  PRESS_AHEAD_OF_X: 5 claims
  X_ONLY: 2 claims
  PRESS_ONLY: 5 claims

Summary: Press and X share consensus on reshoring trends, security motivations, and execution challenges in semiconductor supply chains. Press provides unique specifics on US-Taiwan deals, exact investment figures, and regional reshoring statistics. X highlights emerging capacity shortages, future shortages through 2027, global labor division vulnerabilities, and India-focused shifts not covered in press reporting.



You can also swap in different domain lists. For example, researching a developer tools topic with `TECH_DOMAINS` instead of news outlets:

In [15]:
tech_brief = research(
    "AI coding tools reshaping software development",
    web_domains=TECH_DOMAINS,
)

print(f"Topic: {tech_brief.topic}")
for cluster in tech_brief.clusters:
    print(f"\n{cluster.category} ({len(cluster.claims)} claims)")
    for claim in cluster.claims:
        print(f"  [{claim.source}] {claim.text}")
        print(f"    {claim.url}")
print(f"\nSummary: {tech_brief.executive_summary}")

Topic: AI Coding Tools Reshaping Software Development

CONSENSUS (5 claims)
  [Press] AI coding tools have evolved from autocomplete to agentic systems where developers describe features in plain language and AI handles implementation, reading files, using tools, and running in parallel
    https://www.wired.com/story/claude-code-success-anthropic-business-model/
  [X] Rapid evolution of specialized AI tools like Cursor and Claude Code into full-lifecycle parallel agents handling planning, testing, and shipping
    https://x.com/petergyang/status/1938036266517598307
  [X] Developers are shifting from typing code to directing agents, reviewing outputs, system design, and high-level planning
    https://x.com/kkworld/status/2040396161048322479
  [Press] The surge in AI-written code has increased bugs, security risks, and hard-to-understand code
    https://techcrunch.com/2026/03/09/anthropic-launches-code-review-tool-to-check-flood-of-ai-generated-code/
  [X] Worries about code quality, 

## Conclusion

The same topic looks different depending on where you look. The three-pass pattern (scoped web search, then X search, then structured reconciliation) gives you a repeatable way to surface those differences.

| Design decision | Why it matters |
|---|---|
| `web_search(allowed_domains=[...])` | You choose the sources you trust |
| `x_search()` | Real-time public discourse, no developer account needed |
| `include=["inline_citations"]` | Claims link back to sources so you can verify |
| `response_format=PydanticModel` | Machine-readable output, not just prose |

### Ideas to extend this
- **X handle filtering**: Use `x_search(allowed_x_handles=[...])` to scope social search to specific voices
- **Date ranges**: Use `from_date` and `to_date` on `x_search()` to focus on a specific time window
- **Scheduling**: Run `research()` on a cron and track how the briefing shifts over days